# Predikcia cien elektriny

Predpovedáme **24 hodinových cien na zajtra**. Predpoveď robíme dnes ráno, ešte pred 13:00, keď burza
zajtrajšie ceny ešte nezverejnila. Poznáme teda všetky ceny do dneška vrátane, ale nič zo zajtrajška.

Postup: vstupy → rozdelenie dát podľa času → baseline → modely → porovnanie → analýza chýb →
finálny test → predpoveď na zajtra. Najprv spusti `main.py`, aby databáza bola aktuálna.

> ⚠️ Pred commitom daj v notebooku **Clear All Outputs** – výsledky sú odvodené dáta
> a licencia ich nedovoľuje zverejňovať.

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Chart style: same as analysis.ipynb
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'figure.facecolor': '#fcfcfb',
    'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e',
    'axes.titlecolor': '#0b0b0b',
    'axes.titlesize': 13,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'axes.axisbelow': True,
    'grid.color': '#e1e0d9',
    'grid.linewidth': 0.8,
    'xtick.color': '#898781',
    'ytick.color': '#898781',
    'lines.linewidth': 2,
    'legend.frameon': False,
})
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']  # blue, orange, aqua, yellow - always in this order
label_color = '#52514e'
muted = '#a8a7a0'

## 1. Načítanie dát

In [ ]:
assert os.path.exists('prices.db'), 'prices.db neexistuje - najprv spusti main.py'

conn = sqlite3.connect('prices.db')
df = pd.read_sql('SELECT date, hour, price FROM prices ORDER BY date, hour', conn)
conn.close()

df['time'] = pd.to_datetime(df['date']) + pd.to_timedelta(df['hour'], unit='h')
df = df.set_index('time')[['price']]
price = df['price']  # the raw series, used again at the end for tomorrow's prediction
print(f'Dáta od {df.index.min():%d.%m.%Y} do {df.index.max():%d.%m.%Y %H:%M}')

## 2. Vstupy (features)

Model dostane pre každú hodinu tieto vstupy (X) a učí sa z nich odhadnúť cenu (y):

| vstup | čo to je |
|---|---|
| `hour`, `weekday`, `month`, `is_weekend` | kalendár |
| `lag_24`, `lag_48`, `lag_168` | cena v tú istú hodinu pred 1, 2 a 7 dňami |
| `prev_day_mean` | priemerná cena predchádzajúceho dňa |
| `prev_week_mean` | priemerná cena za predchádzajúcich 7 dní |

**Žiadny vstup nesmie byť zo zajtrajška** (únik dát). Preto tu nie je cena pred hodinou (`lag_1`):
ráno pri predpovedi ešte nepoznáme zajtrajšiu cenu o 12:00, takže ju nemôžeme použiť pri 13:00.
Najbližšia známa cena v tú istú hodinu je dnešná, teda `lag_24`.

`shift(24, freq='h')` posunie ceny o 24 hodín podľa času, nie podľa poradia riadkov,
takže sa nič nepomieša ani pri chýbajúcich dňoch.

In [ ]:
df['hour'] = df.index.hour
df['weekday'] = df.index.dayofweek  # 0 = Monday ... 6 = Sunday
df['month'] = df.index.month
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

# prices known at prediction time: the same hour 1, 2 and 7 days earlier
for lag in [24, 48, 168]:
    df[f'lag_{lag}'] = price.shift(lag, freq='h').reindex(df.index)

# averages of whole days before the predicted day
daily = price.resample('D').mean()
day = df.index.normalize()  # 2026-10-20 13:00 -> 2026-10-20
df['prev_day_mean'] = daily.shift(1).reindex(day).values
df['prev_week_mean'] = daily.rolling(7).mean().shift(1).reindex(day).values

df = df.dropna()  # the first week (no lags yet) and hours next to missing data
features = ['hour', 'weekday', 'month', 'is_weekend',
            'lag_24', 'lag_48', 'lag_168', 'prev_day_mean', 'prev_week_mean']
df[features + ['price']].tail()

## 3. Rozdelenie dát podľa času

Nikdy náhodne – model by sa inak učil aj z budúcnosti.

- **trénovanie** (2023 – 2024): na týchto dátach sa model učí (`fit`)
- **validácia** (2025): na nich porovnávame modely a rozhodujeme, čo funguje
- **test** (2026): použijeme ho **raz, na úplnom konci**, aby bol výsledok poctivý

Trénujeme od roku 2023, teda po energetickej kríze. Či by bolo lepšie trénovať od 2015, vyskúšame v sekcii 10.

In [ ]:
train = df.loc['2023-01-01':'2024-12-31']
val = df.loc['2025-01-01':'2025-12-31']
test = df.loc['2026-01-01':]

X_train, y_train = train[features], train['price']
X_val, y_val = val[features], val['price']
X_test, y_test = test[features], test['price']

pd.DataFrame({
    'od': [part.index.min().date() for part in (train, val, test)],
    'do': [part.index.max().date() for part in (train, val, test)],
    'hodín': [len(part) for part in (train, val, test)],
}, index=['trénovanie', 'validácia', 'test'])

## 4. Baseline

Najjednoduchšie odhady bez učenia. Každý model musí byť lepší, inak nemá zmysel.

- **včera**: zajtra o 13:00 bude cena ako dnes o 13:00 (`lag_24`)
- **pred týždňom**: zajtra bude cena ako v ten istý deň pred týždňom (`lag_168`)

**MAE** = priemerná chyba v €/kWh, **RMSE** viac trestá veľké chyby. Menej = lepšie.

In [ ]:
results = {}


def evaluate(name, y_true, pred):
    """Store MAE and RMSE of one model on the validation data"""
    results[name] = {'MAE': mean_absolute_error(y_true, pred), 'RMSE': root_mean_squared_error(y_true, pred)}


evaluate('baseline: včera', y_val, X_val['lag_24'])
evaluate('baseline: pred týždňom', y_val, X_val['lag_168'])
pd.DataFrame(results).T.round(4)

## 5. Lineárna regresia

Lineárna regresia potrebuje vstupy upraviť:

- **one-hot** pre hodinu, deň v týždni a mesiac: z hodiny 13 spraví stĺpec `hour_13 = 1` a ostatné hodiny 0.
  Inak by si model myslel, že cena s číslom hodiny rovnomerne rastie.
- **škálovanie** (`StandardScaler`) pre čísla: priemer 0, smerodajná odchýlka 1.

`make_pipeline` spojí úpravu a model do jedného objektu. Úprava sa tak „naučí“ len z trénovacích dát
a na validačné sa len použije – žiadny únik dát.

In [ ]:
categorical = ['hour', 'weekday', 'month']
numeric = ['is_weekend', 'lag_24', 'lag_48', 'lag_168', 'prev_day_mean', 'prev_week_mean']

prepare = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical),  # hour 13 -> hour_13 = 1
    ('scale', StandardScaler(), numeric),                              # mean 0, std 1
])
linear = make_pipeline(prepare, LinearRegression())
linear.fit(X_train, y_train)
evaluate('lineárna regresia', y_val, linear.predict(X_val))
pd.DataFrame(results).T.round(4)

## 6. Random Forest a gradient boosting

Stromové modely nepotrebujú one-hot ani škálovanie – stromu je jedno, či je hodina 13 alebo 0,13.
Vedia aj kombinácie, ktoré lineárna regresia nezachytí (napríklad „víkend **a** obed“).

- **Random Forest**: 200 stromov, každý trochu iný, výsledok je ich priemer.
  `min_samples_leaf=5` = list stromu musí mať aspoň 5 hodín, aby sa strom nenaučil dáta naspamäť.
- **Gradient boosting**: stromy idú za sebou a každý opravuje chyby predošlých.

`random_state=0` zaručí, že pri každom spustení vyjde to isté.

In [ ]:
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0)
forest.fit(X_train, y_train)
evaluate('Random Forest', y_val, forest.predict(X_val))

boosting = HistGradientBoostingRegressor(random_state=0)
boosting.fit(X_train, y_train)
evaluate('gradient boosting', y_val, boosting.predict(X_val))
pd.DataFrame(results).T.round(4)

## 7. Porovnanie modelov (validácia 2025)

Stĺpec „lepší ako baseline“ ukazuje, o koľko percent je MAE nižšie ako pri lepšom z dvoch baseline.

In [ ]:
table = pd.DataFrame(results).T.sort_values('MAE')
is_baseline = table.index.str.startswith('baseline')
best_baseline = table.loc[is_baseline, 'MAE'].min()
table['lepší ako baseline'] = (1 - table['MAE'] / best_baseline).map('{:.0%}'.format)

fig, ax = plt.subplots(figsize=(10, 3.5), layout='constrained')
bars = table['MAE'][::-1]  # best at the top
ax.barh(bars.index, bars.values, color=[muted if name.startswith('baseline') else colors[0] for name in bars.index], height=0.6)
for name, value in bars.items():
    ax.annotate(f'{value:.4f}', xy=(value, name), xytext=(4, 0), textcoords='offset points', va='center', color=label_color)
ax.grid(True, axis='x')
ax.grid(False, axis='y')
ax.set_title('MAE na validačných dátach (€/kWh, menej = lepšie)')
plt.show()

table.round(4)

## 8. Ako vyzerá predpoveď

Najlepší model podľa MAE na validácii vs. skutočná cena za posledné 3 týždne roka 2025.

In [ ]:
models = {'lineárna regresia': linear, 'Random Forest': forest, 'gradient boosting': boosting}
best_name = min(models, key=lambda name: results[name]['MAE'])
best = models[best_name]
print('Najlepší model:', best_name)

val_pred = pd.Series(best.predict(X_val), index=X_val.index)
last_weeks = slice(val.index.max() - pd.Timedelta(days=21), None)

fig, ax = plt.subplots(figsize=(10, 4.5), layout='constrained')
ax.plot(y_val.loc[last_weeks], color=colors[0], linewidth=1.5, label='skutočná cena')
ax.plot(val_pred.loc[last_weeks], color=colors[1], linewidth=1.5, label=f'predpoveď ({best_name})')
ax.set_title('Skutočná vs. predpovedaná cena, koniec roka 2025 (€/kWh)')
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.1), ncols=2)  # legend below the chart
plt.show()

## 9. Kde sa model mýli

Priemerná chyba podľa hodiny dňa pre najlepší model a pre baseline „včera“.

In [ ]:
errors = pd.DataFrame({
    best_name: (val_pred - y_val).abs(),
    'baseline: včera': (X_val['lag_24'] - y_val).abs(),
})
by_hour = errors.groupby(X_val['hour']).mean()

fig, ax = plt.subplots()
for (name, series), color in zip(by_hour.items(), colors):
    ax.plot(series.index, series.values, color=color, label=name)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel('hodina')
ax.set_ylim(bottom=0)
ax.set_title('MAE podľa hodiny dňa, validácia 2025 (€/kWh)')
ax.legend(loc='upper left')
plt.show()

errors.groupby(X_val['month']).mean().round(4).rename_axis('mesiac')

**Ktoré vstupy model naozaj používa?** `permutation_importance` náhodne zamieša jeden stĺpec
a zmeria, o koľko sa zhorší MAE. Čím viac sa zhorší, tým je vstup dôležitejší.

In [ ]:
imp = permutation_importance(best, X_val, y_val, scoring='neg_mean_absolute_error',
                             n_repeats=5, random_state=0, n_jobs=-1)
importance = pd.Series(imp.importances_mean, index=features).sort_values()

fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
ax.barh(importance.index, importance.values, color=colors[0], height=0.6)
ax.grid(True, axis='x')
ax.grid(False, axis='y')
ax.set_title(f'O koľko sa zhorší MAE, keď vstup zamiešame ({best_name}, €/kWh)')
plt.show()

## 10. Trénovať od 2015, alebo od 2023?

Ten istý gradient boosting, raz trénovaný na rokoch 2023 – 2024 a raz na 2015 – 2024.
Viac dát nemusí pomôcť, ak sa trh odvtedy zmenil.

In [ ]:
long_train = df.loc['2015-01-01':'2024-12-31']
long_boosting = HistGradientBoostingRegressor(random_state=0).fit(long_train[features], long_train['price'])

pd.Series({
    'trénovanie od 2023': results['gradient boosting']['MAE'],
    'trénovanie od 2015': mean_absolute_error(y_val, long_boosting.predict(X_val)),
}, name='MAE na validácii').round(4)

## 11. Finálny test na roku 2026

Najlepší model natrénujeme znova na trénovacích **aj** validačných dátach (2023 – 2025) s rovnakými
nastaveniami a raz ho vyhodnotíme na roku 2026, ktorý doteraz nevidel. Toto je poctivý výsledok.

In [ ]:
final = clone(best).fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
test_pred = pd.Series(final.predict(X_test), index=X_test.index)

test_table = pd.DataFrame({
    name: {'MAE': mean_absolute_error(y_test, pred), 'RMSE': root_mean_squared_error(y_test, pred)}
    for name, pred in [('baseline: včera', X_test['lag_24']),
                       ('baseline: pred týždňom', X_test['lag_168']),
                       (best_name, test_pred)]
}).T

last_weeks = slice(test.index.max() - pd.Timedelta(days=14), None)
fig, ax = plt.subplots(figsize=(10, 4.5), layout='constrained')
ax.plot(y_test.loc[last_weeks], color=colors[0], linewidth=1.5, label='skutočná cena')
ax.plot(test_pred.loc[last_weeks], color=colors[1], linewidth=1.5, label=f'predpoveď ({best_name})')
ax.set_title('Skutočná vs. predpovedaná cena, posledné 2 týždne (€/kWh)')
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.1), ncols=2)  # legend below the chart
plt.show()

test_table.round(4)

## 12. Predpoveď na zajtra

Vstupy pre 24 hodín dňa po poslednom dni v databáze. Keď `main.py` bežal dnes, je to zajtrajšok.
Na konci nájdeme aj najlacnejšie 4 hodiny za sebou – napríklad na nabíjanie elektromobilu.

In [ ]:
tomorrow = price.index.max().normalize() + pd.Timedelta(days=1)
hours = pd.date_range(tomorrow, periods=24, freq='h')

X_tomorrow = pd.DataFrame(index=hours)
X_tomorrow['hour'] = hours.hour
X_tomorrow['weekday'] = hours.dayofweek
X_tomorrow['month'] = hours.month
X_tomorrow['is_weekend'] = (X_tomorrow['weekday'] >= 5).astype(int)
for lag in [24, 48, 168]:
    X_tomorrow[f'lag_{lag}'] = price.reindex(hours - pd.Timedelta(hours=lag)).values
X_tomorrow['prev_day_mean'] = daily.iloc[-1]          # the last day in the database
X_tomorrow['prev_week_mean'] = daily.iloc[-7:].mean()  # the last 7 days
assert X_tomorrow.notna().all().all(), 'v posledných dňoch chýbajú dáta - spusti main.py'

forecast = pd.Series(final.predict(X_tomorrow[features]), index=hours.hour)

window = forecast.rolling(4).sum()  # price of 4 hours ending at each hour
end = window.idxmin()
start = end - 3

fig, ax = plt.subplots()
ax.bar(forecast.index, forecast.values, color=[colors[1] if start <= h <= end else colors[0] for h in forecast.index], width=0.7)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel('hodina')
ax.set_title(f'Predpoveď na {tomorrow:%d.%m.%Y} (€/kWh), oranžovo najlacnejšie 4 hodiny')
plt.show()

print(f'Najlacnejšie 4 hodiny za sebou: {start}:00 – {end + 1}:00')

## Zhrnutie

Zhrnutie je zámerne bez čísel – odvodené dáta sa podľa licencie nesmú zverejňovať.
Presné hodnoty vypíšu sekcie po spustení (**Run All**).

1. **Všetky tri modely sú lepšie ako baseline, rozdiely medzi nimi sú však malé.** Gradient boosting má
   najmenšie MAE, lineárna regresia zase najmenšie RMSE, teda najmenej veľkých omylov.
2. **Najdôležitejší vstup je cena v tú istú hodinu včera**, potom deň v týždni a cena pred týždňom.
   `is_weekend` model takmer nepoužíva, lebo tú istú informáciu má v `weekday`.
3. **Model sa najviac mýli cez deň a večer**, najviac okolo večernej špičky; v noci sú chyby malé.
   Oproti baseline sa najviac zlepšil ráno. Prudké výkyvy cien model tlmí – z minulých cien sa predpovedať nedajú.
4. **Trénovanie od 2015 a od 2023 dopadlo takmer rovnako**, staršie dáta modelu ani nepomohli, ani neuškodili.
5. **Finálny test (2026) potvrdzuje validáciu.** Chyby sú v roku 2026 väčšie, lebo ceny sú vyššie a viac kolíšu,
   ale náskok modelu pred baseline je podobný ako na validácii – model nie je pretrénovaný.

**Čo by mohlo pomôcť ďalej:** slovenské sviatky, predpoveď počasia a výroby zo slnka a vetra (ENTSO-E),
ladenie nastavení modelu a experiment s hlbokým učením (MLP, LSTM).